In [19]:
from minio import Minio
from minio.error import S3Error
import math
import requests
from requests.auth import HTTPBasicAuth
import json
import zipfile
import time
import os

import jwt
import datetime
import pytz
from datetime import datetime
# Definition of necessary functions

def get_token(text):
    browser=['token":"','","file_stage_in']
    pos=[]
    for n in browser:
        k =text.find(n)
        if k!=-1:
            pos.append(k)
        else:
            break
    if k==-1:
        print('Error in connection')
        return None
    else:
        return text[pos[0]+8:pos[1]]

def get_cpuService(text):
    browser=['cpu":"','","total_memory']
    pos=[]
    for n in browser:
        k=text.find(n)
        if k!=-1:
            pos.append(k)
        else:
            break
    if k==-1:
        print('Error in connection')
        return None
    else:
        return 1000*float(text[pos[0]+6:pos[1]])

def get_memoryService(text):
    browser=['memory":"','Gi']
    pos=[]
    for n in browser:
        k=text.find(n)
        if k!=-1:
            pos.append(k)
        else:
            break
    if k==-1:
        print('Error in connection')
        return None
    else:
        return (float(text[pos[0]+9:pos[1]]))

def connect_to_minio(config):
    MinIO_url = config['url']
    MinIO_access_key = config['access_key']
    MinIO_secret_key = config['secret_key']
    #print(f"Connecting to MinIO at {url_minio} with access key {access_key}")
    return MinIO_url,MinIO_access_key,MinIO_secret_key 

def use_bucket(config):
    bucket_name = config['name']
    folder_prefix = config['folder_prefix']
    #print(f"Using bucket {bucket_name} with folder prefix {folder_prefix}")
    return bucket_name, folder_prefix

def setup_output(config):
    output_file = config['file']
    return output_file

def use_service(config):
    service_name = config['name']
    return service_name

def connect_to_oscar_cluster(config):
    refresh_token=''
    oscar_cluster= config['url']
    if 'username' in config_data.get('oscar_cluster', {}).get('auth_basic', {}):
        username = config['auth_basic']['username']
    if 'password' in config_data.get('oscar_cluster', {}).get('auth_basic', {}):
        password = config['auth_basic']['password']
    if username !="" and password != "":
        basic= True 
    else:
        if 'refresh_token' in config_data.get('oscar_cluster', {}).get('auth_token', {}):
            refresh_token = config['auth_token']['refresh_token']
            if refresh_token !='':
                basic=False
             
    return oscar_cluster,username,password,refresh_token,basic
def new_token(url,refresh_token):
    
    data = {
    'grant_type': 'refresh_token',
    'refresh_token':refresh_token,
    'client_id': 'token-portal',
    'scope': 'openid email profile voperson_id eduperson_entitlement'
}


    response = requests.post(url, data=data)


    if response.status_code == 200:
    
        err=response.status_code
        response_data = response.json()
   
    
        access_token = response_data.get('access_token')
        expires_in = response_data.get('expires_in')

    else:
        print(f"Error: {response.status_code}, {response.text}")
        err=response.status_code

    return access_token, err
def expire_token(token):
    try:
   # Decode the token (without verifying the signature)
        decoded_token = jwt.decode(token, options={"verify_signature": False}, algorithms=["HS256"])
        

        # Extract the 'exp' expiration field
        exp_timestamp = decoded_token.get('exp')

        if exp_timestamp:
        
            #exp_datetime = datetime.datetime.utcfromtimestamp(exp_timestamp)
            exp_datetime = datetime.utcfromtimestamp(exp_timestamp)
            timezone_spain = pytz.timezone('Europe/Madrid')

        # Convert UTC time to Spain time zone
            exp_datetime_spain = exp_datetime.replace(tzinfo=pytz.utc).astimezone(timezone_spain)

            #current_time = datetime.datetime.utcnow()
            current_time = datetime.utcnow()
            current = current_time.replace(tzinfo=pytz.utc).astimezone(timezone_spain)
            k=exp_datetime_spain - current

            if k.total_seconds()/60 <0:
                print("Token has expired")
            
        else:
            print("The token has no expiration date")

    except jwt.ExpiredSignatureError:
        print("The token has already expired.")
    except jwt.InvalidTokenError:
        print("Invalid token.")
    return exp_datetime_spain, k    


In [24]:
with open('config-walton-refresh-token.json', 'r') as config_file:
    config_data = json.load(config_file)

# Take configuration values
MinIO_url,MinIO_access_key,MinIO_secret_key = connect_to_minio(config_data['MinIO'])
bucket_name, folder_prefix = use_bucket(config_data['bucket'])
output_file=setup_output(config_data['output'])
service_name=use_service(config_data['service'])
oscar_cluster, username, password,refresh_token, basic = connect_to_oscar_cluster(config_data['oscar_cluster'])

# Configure the MinIO client
client = Minio(
    MinIO_url,  # MinIO server
    access_key=MinIO_access_key,  
    secret_key=MinIO_secret_key,  
    secure=True  
)

output_path = folder_prefix + output_file

try:
    # List objects in the bucket
    objects = client.list_objects(bucket_name,  prefix=folder_prefix)
    
    # Count the number of objects
    object_list = []
    for obj in objects:
         if obj.object_name.endswith('.jpg'):
            #print(obj.object_name)
            object_list.append(obj.object_name)

    num_imag = len(object_list)
     # Open the file in write mode
    with open(output_file, 'w') as file:
        for obj in object_list:
            #print(f"{obj}\n")
            file.write(f"{obj}\n")

    # Upload the text file to the bucket
    client.fput_object(
        bucket_name, 
        output_path,
        output_file,
        content_type="text/plain"
    )

    print(f"File {output_file} uploaded to {bucket_name}")
except S3Error as exc:
    print("Error occurred: ", exc)
print(f"Total images to proccess: {num_imag}")

File index.txt uploaded to fish-detector
Total images to proccess: 56


In [25]:
service_info = "https://" + oscar_cluster + "/system/services/" + service_name
print(service_info)
# Get token from refresh_token
url = 'https://aai.egi.eu/auth/realms/egi/protocol/openid-connect/token'
token_cluster,err=new_token(url,refresh_token)

    
# GET request via basic authentication or token
if basic:
    response = requests.get(service_info, auth=HTTPBasicAuth(username, password),verify=True)
else:
    headers = {
    'Authorization': "Bearer " + token_cluster
    }
    response = requests.get(service_info, headers=headers,verify=True)
 
# Check the status of the response
if response.status_code == 200:
    resp = response.text
    print(resp)
    # Calculate CPU, Memory and token of the service
    cpu_service = get_cpuService(resp)
    memory_service = get_memoryService(resp)
    print(cpu_service)
    print(memory_service)
    token_service = get_token(resp)
else:
    print(f"Request error: {response.status_code}")
    print("Error message:")
    print(response.text)

https://inference-walton.cloud.imagine-ai.eu/system/services/fish-detector-zip-3
{"name":"fish-detector-zip-3","cluster_id":"","memory":"3Gi","cpu":"1.0","total_memory":"","total_cpu":"","enable_gpu":false,"enable_sgx":false,"image_prefetch":false,"synchronous":{"min_scale":0,"max_scale":0},"delegation":"","rescheduler_threshold":0,"log_level":"DEBUG","image":"dialdroid/fish-detector:zip-2","alpine":false,"token":"3ec1897407976c07e837e0b8f8ab7d3a552f590fe5f283183409cc4ce5c51f90","file_stage_in":false,"input":[],"output":[{"storage_provider":"minio.default","path":"fish-detector-output/output"}],"script":"#!/bin/sh\n\nOUTPUT_FILE=\"$TMP_OUTPUT_DIR/$FILE_NAME\"\nJSON=$(cat \"$INPUT_FILE_PATH\")\necho $JSON\n\nZIP_FILE=$(echo \"$JSON\" | jq -r '.zip')\n\npython3 fish_detector.py -i \"$BUCKET_DIR/$ZIP_FILE\" -o \"$OUTPUT_FILE\"\n\necho  $?","expose":{"min_scale":0,"max_scale":0,"cpu_threshold":0,"rewrite_target":false,"nodePort":0,"default_command":false,"set_auth":false},"environment":{"V

In [26]:
# Initialize CPU and Memory allocation variables
cpu_Alloc=0
cpu_invoke=0
memory_Alloc=0
memory_invoke=0

# Ensure the URL is properly constructed (if `oscar_cluster` does not have "https://")
if not oscar_cluster.startswith("https://"):
    url_status = "https://" + oscar_cluster + "/system/status"
else:
    url_status = oscar_cluster + "/system/status"

# Make the GET request with basic authentication or token authentication
try:
    if basic:
        # Use basic authentication if enabled
        response = requests.get(url_status, auth=HTTPBasicAuth(username, password), verify=False)
    else:
        # Use token authentication if enabled
        headers = {
            'Authorization': "Bearer " + token_cluster
        }
        response = requests.get(url_status, headers=headers, verify=True)

    # Check the status of the response
    if response.status_code == 200:
        # Convert the response to JSON
        try:
            data = response.json()

            # Ensure the response is a dict object
            if isinstance(data, dict):
                nodos = len(data['detail'])
                data = data['detail']
                
                # Iterate over each object in the array, except the front node
                if nodos >= 1:
                    for obj in data:
                        cpu_Alloc=(int(obj['cpuCapacity']))*0.8 - int(obj['cpuUsage'])
                        cpu_invoke += int((cpu_Alloc/cpu_service))
                        memory_Alloc=(int(obj['memoryCapacity'])*0.8) - int(obj['memoryUsage'])
                        memory_invoke += int((memory_Alloc/(1000000000*memory_service)))
            else:
                print("The response is not a JSON array of objects.")
        
        except ValueError as e:
            print("Error converting the response to JSON:", e)
    else:
        print(f"Request error: {response.status_code}")
        print("Error message:")
        print(response.text)

except requests.exceptions.RequestException as e:
    print(f"Connection error: {e}")

print(f"CPU invocations: {cpu_invoke}")
print(f"Memory invocations: {memory_invoke}")

CPU invocations: 39
Memory invocations: 36


In [27]:
# Search for the CPU needed to run the service defined in its creation (FDL)
if not oscar_cluster.startswith("https://"):
    service_info = "https://" + oscar_cluster + "/system/services/" + service_name
else:
    service_info = oscar_cluster + "/system/services/" + service_name
    
# GET request via basic authentication or token
if basic:
    response = requests.get(service_info, auth=HTTPBasicAuth(username, password),verify=True)
else:
    response = requests.get(service_info, headers=headers,verify=True)

# Check the status of the response
if response.status_code == 200:
    resp = response.text
    # Calculate CPU, Memory and token of the service
    cpu_service = get_cpuService(resp)
    memory_service = get_memoryService(resp)  # take 80% of the memory so as not to completely saturate the cluster
    token_service = get_token(resp)
else:
    print(f"Request error: {response.status_code}")
    print("Error message:")
    print(response.text)

# Min value between invocations by CPU and by memory
cant_invoke = min(cpu_invoke, memory_invoke)
print(f"Invocations: {cant_invoke}")

# Calculate the number of images per invocationç
num_imag=36
resto = (num_imag) % cant_invoke
img_invoke = int(num_imag / cant_invoke)
print(f"Images per invocation: {img_invoke}")

Invocations: 36
Images per invocation: 1


In [28]:
output_bucket = bucket_name

if basic:
    headers = {    
    'Authorization': "Bearer " + token_service,
    'Content-Type': 'application/json',
}
else:
    headers = {
    'Authorization': "Bearer " + token_cluster,
    'Content-Type': 'application/json',
    }
    
# Ensure the URL is properly constructed (if `oscar_cluster` does not have "https://")
if not oscar_cluster.startswith("https://"):
    url_invoke = "https://" + oscar_cluster + "/job/" + service_name
else:
    url_invoke = oscar_cluster + "/job/" + service_name

end=0
start=0
# Range of images to process (start-end)
t1=time.time()
for i in range(cant_invoke):
    t=time.time()
    # Range of images
    start = end+1
    end = end+ img_invoke
    if i < resto:
        end = end+1
    name_zip=str(i+1)+".zip"

    data = {
        "zip": name_zip
        
    }
    
    # Create the .zip
    list=object_list[int(start)-1:int(end)]
    zip_file_name = str(start)+".zip"
    output_path = "zip/" + name_zip
    
    print(name_zip)
    local_files=[]
    try:
        t2=time.time()
        time.sleep(2)
        for obj in list:
            
            local_path = f"/tmp/{obj}"
            local_files.append(local_path)
            client.fget_object(bucket_name, obj, local_path)
        
        t21=time.time()
        print(f"Download images from the bucket {round(t21-t2,2)}") 
     
        # Create the .zip
        with zipfile.ZipFile(zip_file_name, "w") as zipf:
            for file in local_files:
                zipf.write(file, arcname=os.path.basename(file))
        print(f"ZIP file '{name_zip}' successfully created.")
        t3=time.time()
        print(f"Crear el zip {t3-t21}") 
    # Upload to bucket the zip file
        client.fput_object(
            output_bucket, 
            output_path,
            zip_file_name
          )
        t4=time.time()
        print(f" Upload to bucket {round(t4-t3,2)} seconds")    
    
   # client.fput_object(output_bucket, zip_file_name, zip_file_name)
        print(f"{name_zip} successfully uploaded to the bucket {output_bucket}")
    
    finally:
      
    # Clean up temporary files
        for file in local_files:
            if os.path.exists(file):
                os.remove(file)
        if os.path.exists(zip_file_name):
                os.remove(zip_file_name)
    
    print(f"Start value: {start}")
    print(f"End value: {end}")
    print(f"Invocation {i + 1} to the service")
    print(url_invoke)
    
    #Check if the token has expired (generate a new token)
    expired,e =expire_token(token_cluster)
    
    if int(e.total_seconds()/60) < 5: #  to generate new_token for 5 min to expired
        token_cluster,err=new_token(url,refresh_token)
        print(token_cluster)
        if basic:
            headers = {    
                  'Authorization': "Bearer " + token_service,
                  'Content-Type': 'application/json',
            }
        else:
            headers = {
            'Authorization': "Bearer " + token_cluster,
            'Content-Type': 'application/json',
           }
    
    
    
    try:
        response = requests.post(url_invoke, headers=headers, json=data,verify=True)
        print(response.text)
        print(response.status_code)
        if response.status_code == 200 or response.status_code == 201:
            print("Services OK")
        else:
            print(response.text)
    except Exception as ex:
        print("Error running service: ", ex)
        print(response.text)
    
    t1=time.time()
    print(f"Total time of service launch: {round(t1-t,2)} seconds")
    
    # time between invocations
    time.sleep(5)
    
t=time.time()
print(f"Total time of the execution process: {round(t-t1,2)} seconds")

    

1.zip
Download images from the bucket 2.34
ZIP file '1.zip' successfully created.
Crear el zip 0.006504535675048828
 Upload to bucket 0.23 seconds
1.zip successfully uploaded to the bucket fish-detector
Start value: 1
End value: 1
Invocation 1 to the service
https://inference-walton.cloud.imagine-ai.eu/job/fish-detector-zip-3

201
Services OK
Total time of service launch: 4.07 seconds
2.zip
Download images from the bucket 2.3
ZIP file '2.zip' successfully created.
Crear el zip 0.0010676383972167969
 Upload to bucket 0.26 seconds
2.zip successfully uploaded to the bucket fish-detector
Start value: 2
End value: 2
Invocation 2 to the service
https://inference-walton.cloud.imagine-ai.eu/job/fish-detector-zip-3

201
Services OK
Total time of service launch: 3.77 seconds
3.zip
Download images from the bucket 2.41
ZIP file '3.zip' successfully created.
Crear el zip 0.0019450187683105469
 Upload to bucket 0.56 seconds
3.zip successfully uploaded to the bucket fish-detector
Start value: 3
End v

In [48]:
output_bucket = bucket_name

if basic:
    headers = {    
    'Authorization': "Bearer " + token_service,
    'Content-Type': 'application/json',
}
else:
    headers = {
    'Authorization': "Bearer " + token_cluster,
    'Content-Type': 'application/json',
    }
    
# Ensure the URL is properly constructed (if `oscar_cluster` does not have "https://")
if not oscar_cluster.startswith("https://"):
    url_invoke = "https://" + oscar_cluster + "/job/" + service_name
else:
    url_invoke = oscar_cluster + "/job/" + service_name

end=0
start=0
# Range of images to process (start-end)
t5=time.time()
for i in range(cant_invoke):
    t=time.time()
    # Range of images
    start = end+1
    end = end+ img_invoke
    if i < resto:
        end = end+1
    name_zip=str(i+1)+".zip"

    data = {
        "zip": name_zip
        
    }
    print(data)
    # Create the .zip
    list=object_list[int(start)-1:int(end)]
    zip_file_name = str(start)+".zip"
    output_path = "zip/" + name_zip
    
    print(name_zip)
    local_files=[]
    try:
        t4=time.time()
        time.sleep(2)
        for obj in list:
            
            local_path = f"/tmp/{obj}"
            local_files.append(local_path)
            client.fget_object(bucket_name, obj, local_path)
        
        t41=time.time()
        print(f"Download images from the bucket {round(t41-t4,2)}") 
     
        # Create the .zip
        with zipfile.ZipFile(zip_file_name, "w") as zipf:
            for file in local_files:
                zipf.write(file, arcname=os.path.basename(file))
        print(f"ZIP file '{name_zip}' successfully created.")
        t2=time.time()
        print(f"Crear el zip {t2-t41}") 
    # Upload to bucket the zip file
        client.fput_object(
            output_bucket, 
            output_path,
            zip_file_name
          )
        t3=time.time()
        print(f" Upload to bucket {round(t3-t2,2)}")    
    
   # client.fput_object(output_bucket, zip_file_name, zip_file_name)
        print(f"{name_zip} successfully uploaded to the bucket {output_bucket}")
    
    finally:
      
    # Clean up temporary files
        for file in local_files:
            if os.path.exists(file):
                os.remove(file)
        if os.path.exists(zip_file_name):
                os.remove(zip_file_name)
    
    print(f"Start value: {start}")
    print(f"End value: {end}")
    print(f"Invocation {i + 1} to the service")
    print(url_invoke)
    """
    try:
        response = requests.post(url_invoke, headers=headers, json=data,verify=True)
        print(response.text)
        print(response.status_code)
        if response.status_code == 200:
            print("Services OK")
        else:
            print(response.text)
    except Exception as ex:
        print("Error running service: ", ex)
        print(response.text)
    """
    t1=time.time()
    print(f"Total time of service launch: {round(t1-t,2)} seconds")
    
t=time.time()
print(f"Total time of the execution process: {round(t-t5,2)} seconds")

    

{'zip': '1.zip'}
1.zip
descargar imagenes 21.458290815353394
Archivo ZIP '1.zip' creado exitosamente.
Crear el zip 0.25125765800476074
Subida al bucket 2.963343620300293
1.zip subido exitosamente al bucket fish-detector
Start value: 1
End value: 112
Invocation 1 to the service
https://inference-walton.cloud.imagine-ai.eu/job/fish-zip
Tiempo total de lanzamiento de invocacion 24.689903736114502
{'zip': '2.zip'}
2.zip
descargar imagenes 21.272411584854126
Archivo ZIP '2.zip' creado exitosamente.
Crear el zip 0.3386821746826172
Subida al bucket 2.97708797454834
2.zip subido exitosamente al bucket fish-detector
Start value: 113
End value: 224
Invocation 2 to the service
https://inference-walton.cloud.imagine-ai.eu/job/fish-zip
Tiempo total de lanzamiento de invocacion 24.618128538131714
{'zip': '3.zip'}
3.zip
descargar imagenes 21.750040531158447
Archivo ZIP '3.zip' creado exitosamente.
Crear el zip 0.15879106521606445
Subida al bucket 2.271512031555176
3.zip subido exitosamente al bucket 